<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/13-neural-networks.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 13 — Networks From First Principles

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/networks-from-first-principles/).

A perceptron, a problem it cannot solve, and backpropagation derived by hand and checked numerically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 1. The perceptron, 1958

In [ ]:
def perceptron(X, y, epochs=20, rate=0.1):
    """Rosenblatt's rule: when wrong, nudge the weights toward the example."""
    w, b = np.zeros(X.shape[1]), 0.0
    for epoch in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            if yi * (xi @ w + b) <= 0:                       # misclassified (labels are -1 and +1)
                w, b = w + rate * yi * xi, b + rate * yi
                mistakes += 1
        if mistakes == 0:
            return w, b, epoch + 1
    return w, b, epochs

rng = np.random.default_rng(0)
A = np.vstack([rng.normal([-2, -2], 0.8, (50, 2)), rng.normal([2, 2], 0.8, (50, 2))])
ya = np.array([-1] * 50 + [1] * 50)
w, b, epochs = perceptron(A, ya)
print(f'two separable blobs: converged after {epochs} epochs, w = {np.round(w, 2)}, b = {b:.2f}')

In [ ]:
XOR = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([-1, 1, 1, -1])
w, b, epochs = perceptron(XOR, y_xor, epochs=1000)
pred = np.sign(XOR @ w + b)
print(f'XOR: after {epochs} epochs, still wrong on {np.sum(pred != y_xor)} of 4 points')

## 2. Layers without a bend collapse into one layer

In [ ]:
rng = np.random.default_rng(1)
W1, b1 = rng.normal(size=(2, 8)), rng.normal(size=8)
W2, b2 = rng.normal(size=(8, 1)), rng.normal(size=1)
x = rng.normal(size=(5, 2))
two_layers = (x @ W1 + b1) @ W2 + b2
one_layer = x @ (W1 @ W2) + (b1 @ W2 + b2)
print('two linear layers give the same output as one:', np.allclose(two_layers, one_layer))

## 3. Activation functions

In [ ]:
def relu(z):
    return np.maximum(0, z)

z = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
print('z      ', z)
print('sigmoid', np.round(sigmoid(z), 3))
print('tanh   ', np.round(np.tanh(z), 3))
print('relu   ', relu(z))

## 4. A forward pass through a two-layer network

In [ ]:
def forward(X, params):
    """Return every intermediate value, because backpropagation needs them."""
    W1, b1, W2, b2 = params
    z1 = X @ W1 + b1               # pre-activation of the hidden layer
    h = np.tanh(z1)                # hidden activations
    z2 = h @ W2 + b2               # one output per example
    p = sigmoid(z2)                # probability of class 1
    return z1, h, z2, p

def init(n_in, n_hidden, seed=0):
    rng = np.random.default_rng(seed)
    return [rng.normal(0, 0.5, (n_in, n_hidden)), np.zeros(n_hidden),
            rng.normal(0, 0.5, (n_hidden, 1)), np.zeros(1)]

params = init(2, 2, seed=0)
y01 = (y_xor + 1) // 2                                          # 0/1 labels for log loss
_, _, _, p = forward(XOR, params)
print('untrained network on XOR:', np.round(p.ravel(), 3), '  wanted:', y01)

## 5. Backpropagation, by the chain rule

In [ ]:
def log_loss(p, y):
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def backward(X, y, params):
    W1, b1, W2, b2 = params
    z1, h, z2, p = forward(X, params)
    n = len(y)
    delta2 = (p - y[:, None]) / n                 # dL/dz2: chapter 4's gradient, once again
    dW2 = h.T @ delta2                            # dL/dW2
    db2 = delta2.sum(axis=0)
    delta1 = (delta2 @ W2.T) * (1 - h ** 2)       # dL/dz1: pass the error back through W2, then through tanh'
    dW1 = X.T @ delta1
    db1 = delta1.sum(axis=0)
    return [dW1, db1, dW2, db2]

def loss_of(params):
    return log_loss(forward(XOR, params)[3].ravel(), y01)

grads = backward(XOR, y01, params)
print('gradient of the loss with respect to W1:\n', np.round(grads[0], 4))

## 6. Check it numerically before trusting it

In [ ]:
def numerical_grads(params, h=1e-5):
    out = []
    for i, P in enumerate(params):
        g = np.zeros_like(P)
        for idx in np.ndindex(P.shape):
            plus = [q.copy() for q in params]; plus[i][idx] += h
            minus = [q.copy() for q in params]; minus[i][idx] -= h
            g[idx] = (loss_of(plus) - loss_of(minus)) / (2 * h)
        out.append(g)
    return out

num = numerical_grads(params)
for name, a, b in zip(['W1', 'b1', 'W2', 'b2'], grads, num):
    print(f'{name}: largest difference between backprop and finite differences {np.max(np.abs(a - b)):.2e}')

## 7. Training on XOR

In [ ]:
def train(X, y, params, rate, epochs, report=()):
    for epoch in range(1, epochs + 1):
        grads = backward(X, y, params)
        params = [P - rate * g for P, g in zip(params, grads)]
        if epoch in report:
            print(f'epoch {epoch:>5}  loss {log_loss(forward(X, params)[3].ravel(), y):.4f}')
    return params

params = train(XOR, y01, init(2, 2, seed=0), rate=1.0, epochs=5000, report=(1, 100, 1000, 5000))
_, h, _, p = forward(XOR, params)
print('trained network on XOR:', np.round(p.ravel(), 3), '  wanted:', y01)
print('what the two hidden units computed for the four inputs:\n', np.round(h, 2))

## 8. The same code on a harder shape

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression

Xm, ym = make_moons(300, noise=0.2, random_state=0)
Xm_t, ym_t = make_moons(3000, noise=0.2, random_state=1)

for hidden in [1, 2, 4, 16]:
    params = train(Xm, ym, init(2, hidden, seed=0), rate=0.5, epochs=3000)
    acc = np.mean((forward(Xm_t, params)[3].ravel() > 0.5) == ym_t)
    n_params = 2 * hidden + hidden + hidden + 1
    print(f'{hidden:>2} hidden units ({n_params:>3} weights): accuracy on new points {acc:.3f}')
print(f'logistic regression (3 weights)   : accuracy on new points {LogisticRegression().fit(Xm, ym).score(Xm_t, ym_t):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 150), np.linspace(-1, 1.5, 150))
grid = np.c_[xx.ravel(), yy.ravel()]
for ax, hidden in zip(axes, [1, 2, 16]):
    params = train(Xm, ym, init(2, hidden, seed=0), rate=0.5, epochs=3000)
    P = forward(grid, params)[3].reshape(xx.shape)
    ax.contourf(xx, yy, P, levels=[0, 0.5, 1], alpha=0.2, cmap='coolwarm')
    ax.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap='coolwarm', s=10, edgecolor='k', linewidth=0.3)
    ax.set_title(f'{hidden} hidden unit{"s" if hidden > 1 else ""}')
plt.tight_layout(); plt.show()

## 9. The one line you would ship

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), activation='tanh', solver='lbfgs', max_iter=2000, random_state=0))
mlp.fit(Xm, ym)
print(f'MLPClassifier, one hidden layer of 16: accuracy on new points {mlp.score(Xm_t, ym_t):.3f}')

## 10. A real task: the digits

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(digits.data, digits.target, test_size=0.3, random_state=0)
for layers in [(), (32,), (64, 64)]:
    m = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=layers, max_iter=2000, random_state=0)).fit(Xd_tr, yd_tr)
    n_w = sum(c.size for c in m[-1].coefs_) + sum(i.size for i in m[-1].intercepts_)
    label = 'no hidden layer (logistic regression)' if layers == () else f'hidden layers {layers}'
    print(f'{label:<40} {n_w:>6} weights   test accuracy {m.score(Xd_te, yd_te):.3f}')

## 11. How much data a network wants

In [ ]:
for n in [50, 200, 800, len(yd_tr)]:
    idx = rng.choice(len(yd_tr), n, replace=False)
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xd_tr[idx], yd_tr[idx])
    nn = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=2000, random_state=0)).fit(Xd_tr[idx], yd_tr[idx])
    print(f'{n:>5} training digits: logistic regression {lr.score(Xd_te, yd_te):.3f}   network {nn.score(Xd_te, yd_te):.3f}')

## Try it yourself

1. In section 7, change `init(2, 2, seed=0)` to each of the seeds 1 through 9. How many of the ten runs solve XOR? That is chapter 3's many valleys.
2. Replace `np.tanh` with `relu` in `forward`, and `(1 - h ** 2)` with `(z1 > 0)` in `backward`. Re-run the gradient check. Does it still pass, and does XOR still train?
3. In section 10, add a third hidden layer. Does accuracy improve, and what happens to training time?